# Credit Risk Guru — the `ear` package, reasoning at runtime

This notebook uses [`ear`](../ear/__init__.py), where every class, field
and message is natively English, and where every judgment-laden stage —
`Policy` compliance, `Discoverer` relevance, `Reasoner`'s decision,
`Explainer`'s prose — reasons in natural language against a live Claude
model via DSPy. The whole point of `ear` is that **the author writes no
logic**: capabilities, governance and applicants are all expressed as
prompts and reasoned out by the runtime, never hand-coded.

Three things distinguish this notebook:

1. **`Skill` is a prompt, not code.** The score/DTI banding and risk
   grading rules are written in plain English inside each `Skill`; no
   scoring function or decision grid is hand-written. The runtime threads
   the stacked skill prompts into the `Reasoner`, which bands and grades
   as part of reasoning.
2. **`Policy` is written in plain English**, e.g. *"The applicant's
   debt-to-income ratio must not exceed 0.45"* — judged by the LLM, with
   no hardcoded boolean expression anywhere.
3. **Applicant scenarios are not hardcoded.** There is no literal Python
   dict of "Asha Rao, score 760, DTI 0.22" anywhere below. A DSPy
   signature (`GenerateApplicantScenario`) reasons each applicant into
   existence from a one-line steer at call time.

Because every stage is driven by reasoning, this notebook needs
`ANTHROPIC_API_KEY` set to run end to end. (The `ear` package itself keeps
a dependency-free fallback on each stage so `import ear` and the test
suite work with no model — but this demo shows the real, reasoning path.)

| `ear` class | Credit risk role |
| --- | --- |
| `Skill` | One scoring capability written as a prompt (score band, DTI band, risk grade) |
| `Persona` | The credit analyst persona carrying out the judgement |
| `Workflow` | The underwriting workflow ordering personas |
| `Process` | "Personal Loan Underwriting", with a `description` the `Discoverer` reasons over |
| `Policy` | Hard underwriting / fair-lending floors, written in plain English |
| `Runtime` | Runs the full Governor → ... → Adapter pipeline every cycle |
| `ModelBinding` | The live Claude binding every LLM-judged stage reasons against |
| `Evidence` | The audit trail behind one decision |
| `Memory` / `Experience` / `Adaptation` | What happened / the pattern / the distilled lesson |

## 0. Imports

Read the Anthropic API key only from the environment — never hardcode a
key in a notebook cell or commit one to a file. This notebook reasons at
every stage, so `ANTHROPIC_API_KEY` must be set for it to generate
applicants and run end to end.

In [ ]:
import os

from ear import (
    Intent,
    ModelBinding,
    Persona,
    Policy,
    Process,
    Runtime,
    Skill,
    Workflow,
)

print("ear package ready --", "ANTHROPIC_API_KEY is set" if os.environ.get("ANTHROPIC_API_KEY") else "no ANTHROPIC_API_KEY set (deterministic fallbacks will run)")

## 1. `Skill` — scoring capabilities, written as prompts

Each `Skill` is a **prompt**, not a Python function. The banding and
grading rules live in plain English inside the skill; the user writes no
scoring code. The persona stacks these skills, and `Runtime.reason()`
threads the stacked skill prompts into the `Reasoner`, so the score tier,
DTI band and risk grade are derived as part of underwriting rather than
precomputed by a hardcoded decision table.

In [ ]:
score_skill = Skill(
    name="credit_score_tier",
    description="Bands a FICO score",
    prompt=(
        "Band the applicant's FICO credit score into a tier: "
        "720 or above is 'prime'; 660 to 719 is 'near-prime'; "
        "620 to 659 is 'subprime'; below 620 is 'deep-subprime'."
    ),
)
dti_skill = Skill(
    name="dti_tier",
    description="Bands a debt-to-income ratio",
    prompt=(
        "Band the applicant's debt-to-income ratio: "
        "0.30 or below is 'low'; above 0.30 up to and including 0.45 is "
        "'moderate'; above 0.45 is 'high'."
    ),
)
grade_skill = Skill(
    name="risk_grade",
    description="Combines score + DTI tiers into a risk grade",
    prompt=(
        "Assign a single risk grade from A (strongest) to E (weakest) from "
        "the credit-score tier and the debt-to-income band. Start from the "
        "tier: prime=A, near-prime=B, subprime=C, deep-subprime=D. If the "
        "debt-to-income band is 'high', drop the grade by one letter; "
        "otherwise keep it. So prime/high=B and deep-subprime/high=E."
    ),
)

for skill in (score_skill, dti_skill, grade_skill):
    print(f"- {skill.name}: {skill.instruction()}")

## 2. `Persona`, `Workflow`, `Process`

`Process.description` is new relative to the original package — it gives
the `Discoverer` a natural-language description to reason over, instead of
only the bare process name.

In [ ]:
credit_risk_guru = Persona(
    name="Credit Risk Guru",
    instructions=(
        "Underwrite personal loans conservatively. Band every applicant's "
        "credit score and debt-to-income ratio, then assign a risk grade "
        "before any approval reasoning happens."
    ),
)
credit_risk_guru.add_skill(score_skill).add_skill(dti_skill).add_skill(grade_skill)

underwriting_workflow = Workflow(name="Underwriting Workflow")
underwriting_workflow.add_persona(credit_risk_guru)

loan_underwriting = Process(
    name="Personal Loan Underwriting",
    description=(
        "Reviews a personal loan applicant's credit score, debt-to-income "
        "ratio, income and existing defaults to decide whether to approve "
        "the requested loan amount."
    ),
)
loan_underwriting.add_workflow(underwriting_workflow)

print(loan_underwriting.name, "->", loan_underwriting.description)

## 3. `Policy` — underwriting floors, written in plain English

Every `Policy` is a single plain-English `statement` and nothing else —
no coded boolean expression. When a `ModelBinding` is active the
`Governor` has the LLM judge each statement against the applicant's
context, exactly the way governance would be written for a human
reviewer. With no model configured a statement-only policy is treated as
*not applicable* — it cannot be machine-evaluated without reasoning — so
the notebook still runs end to end; it simply does not gate until a mind
is attached.

In [ ]:
policies = [
    Policy(
        name="Minimum Credit Score",
        statement="The applicant's FICO credit score must be at least 620.",
    ),
    Policy(
        name="Debt-to-Income Ceiling",
        statement="The applicant's debt-to-income ratio must not exceed 0.45.",
    ),
    Policy(
        name="No Active Defaults",
        statement="The applicant must have no existing loan defaults on record.",
    ),
    Policy(
        name="Loan Amount Cap",
        statement="The requested personal loan amount must not exceed $75,000.",
    ),
]

for policy in policies:
    print(f"- {policy.name}: {policy.statement}")

## 4. `ModelBinding` — give the Guru a real mind, from the environment only

Reads `ANTHROPIC_API_KEY` exclusively via `os.environ` at call time —
`ModelBinding.resolve_api_key()` never has a key passed to it as a
literal. The binding is what every judgment-laden stage reasons against;
without it the reasoning-driven scenario generator below has nothing to
reason with, so a key is required to run the rest of the notebook.

In [ ]:
model_binding = None
if os.environ.get("ANTHROPIC_API_KEY"):
    model_binding = ModelBinding(provider="anthropic", model=os.environ.get("ANTHROPIC_TEST_MODEL", "claude-haiku-4-5"))
    model_binding.activate()
    print(f"ModelBinding attached -- reasoning against {model_binding.model_id}.")
else:
    print(
        "No ANTHROPIC_API_KEY in the environment -- this notebook reasons at "
        "every stage, so the applicant generator below will stop and ask for a "
        "key rather than fall back to hardcoded data."
    )

## 5. `Runtime` — assemble the pipeline

`Runtime` wires in all nineteen pipeline stages itself; we only register
the process, the policies, and the `ModelBinding`.

In [ ]:
runtime = Runtime(name="Credit-Risk-Guru-Runtime", model_binding=model_binding)
runtime.add_process(loan_underwriting)
for policy in policies:
    runtime.add_policy(policy)

print(runtime.name)
print("Processes:", [p.name for p in runtime.processes])
print("Policies: ", [p.name for p in runtime.policies])

## 6. Generating applicant scenarios — reasoned into existence, never a hardcoded list

`GenerateApplicantScenario` is a DSPy signature that reasons one applicant
into existence from a one-line plain-English steer (`portfolio_brief`),
fresh on every call — there is no literal applicant dict and no procedural
fallback anywhere in this notebook. Inventing a realistic applicant is a
judgment task, so it runs against the active model; with no model it stops
and asks for a key rather than fabricating applicants from hardcoded
ranges.

In [ ]:
import dspy


class GenerateApplicantScenario(dspy.Signature):
    """Invent one realistic personal-loan applicant scenario for an
    underwriting demo. Vary the credit profile and narrative each time so
    repeated calls produce a diverse portfolio, not a repeated template."""

    portfolio_brief: str = dspy.InputField(
        desc="A one-line steer on what kind of applicant to invent next"
    )
    applicant_name: str = dspy.OutputField(desc="A plausible full name")
    credit_score: int = dspy.OutputField(desc="A FICO score between 300 and 850")
    debt_to_income_ratio: float = dspy.OutputField(desc="A debt-to-income ratio between 0.0 and 1.0")
    annual_income: float = dspy.OutputField(desc="Annual income in US dollars")
    loan_amount: float = dspy.OutputField(desc="Requested personal loan amount in US dollars")
    existing_defaults: int = dspy.OutputField(desc="Number of existing loan defaults on record")
    narrative: str = dspy.OutputField(desc="One sentence on why this applicant is applying")


# Plain-English steers -- prompts, not data. The applicant behind each is
# reasoned into existence by the LLM at call time; nothing here is a fixed
# applicant record.
portfolio_briefs = [
    "a clean prime-tier applicant who should comfortably clear every policy",
    "a borderline applicant hovering right at the debt-to-income ceiling",
    "a subprime applicant who has missed payments before but carries no active defaults",
    "an applicant whose debt-to-income ratio breaches the ceiling and must be blocked before reasoning starts",
    "a high-income applicant requesting a large loan near the policy cap",
]


def generate_applicant_scenario(brief: str, model_binding) -> dict:
    """Reason one applicant scenario into existence from a one-line steer,
    via the LLM. There is no hardcoded applicant data and no procedural
    fallback -- inventing a realistic applicant is a judgment task, so it
    runs against the active model."""
    if model_binding is None or model_binding.lm is None:
        raise RuntimeError(
            "Applicant generation needs an active ModelBinding -- set "
            "ANTHROPIC_API_KEY so the scenarios are reasoned, not hardcoded."
        )
    program = dspy.Predict(GenerateApplicantScenario)
    with dspy.context(lm=model_binding.lm):
        prediction = program(portfolio_brief=brief)
    return {
        "applicant_name": prediction.applicant_name,
        "credit_score": int(prediction.credit_score),
        "debt_to_income_ratio": float(prediction.debt_to_income_ratio),
        "annual_income": float(prediction.annual_income),
        "loan_amount": float(prediction.loan_amount),
        "existing_defaults": int(prediction.existing_defaults),
        "narrative": prediction.narrative,
    }


scenarios = [generate_applicant_scenario(brief, model_binding) for brief in portfolio_briefs]
for scenario in scenarios:
    print(f"- {scenario['applicant_name']}: score={scenario['credit_score']}, DTI={scenario['debt_to_income_ratio']}, loan=${scenario['loan_amount']:,.0f} -- {scenario['narrative']}")

## 7. Turning a generated scenario into an `Intent`

No scoring code runs here. The raw applicant facts go straight into the
intent's context — the score tier, DTI band and risk grade are **not**
precomputed. Because the Guru's scoring `Skill`s are stacked on its
persona and `Runtime.reason()` threads them into the `Reasoner`, the
banding and grading happen *inside* reasoning, against the same
plain-English rules a human underwriter would apply.

In [ ]:
def scenario_to_intent(scenario: dict) -> Intent:
    context = {
        "credit_score": scenario["credit_score"],
        "debt_to_income_ratio": scenario["debt_to_income_ratio"],
        "annual_income": scenario["annual_income"],
        "loan_amount": scenario["loan_amount"],
        "existing_defaults": scenario["existing_defaults"],
    }
    text = (
        f"Underwrite a ${scenario['loan_amount']:,.0f} personal loan for "
        f"{scenario['applicant_name']}. Band the credit score and "
        f"debt-to-income ratio, assign a risk grade, then decide. "
        f"{scenario['narrative']}"
    )
    return Intent(text=text, context=context)


def show_decision(name: str, entry) -> None:
    print(f"Applicant: {name}")
    print(f"  Decision : {entry.decision}")
    print(f"  Basis    : {entry.evidence.basis}")
    print(f"  Plan     : {entry.evidence.sources['plan']}")
    print(f"  Policies checked: {entry.evidence.sources['policies_checked']}")
    print(f"  Explanation: {entry.evidence.sources['explanation']}")

## 8. Running the generated portfolio through `Runtime.reason()`

`Governor` checks every `Policy` first — before `Discoverer` even
discovers a process — so when a `ModelBinding` is active, an applicant
whose debt-to-income ratio breaches the ceiling never reaches reasoning
at all: `Runtime.reason()` raises `PermissionError` immediately and no
`Memory` entry is written. With no model configured the plain-English
policies are not applicable, so every applicant instead proceeds to the
deterministic fallback decision.

In [ ]:
for scenario in scenarios:
    intent = scenario_to_intent(scenario)
    try:
        runtime.reason(intent)
        show_decision(scenario["applicant_name"], runtime.memory.working[-1])
    except PermissionError as exc:
        print(f"Applicant: {scenario['applicant_name']}")
        print(f"  Rejected before reasoning even started: {exc}")
    print()

print(f"Total successful cycles observed by Experience: {runtime.experience.observations}")

## 9. Inspecting `Memory`, `Experience` and `Adaptation`

`Adapter`'s default `adapt_every=5` means a fifth successful cycle (if
reached) triggers `AdaptationBank` to distill one new lesson from
`Experience`'s aggregated pattern — not on every single call.

In [ ]:
print("=== Memory (what happened) ===")
print(runtime.memory.context_window())

print("\n=== Experience (the pattern across repeated cycles) ===")
print(runtime.experience.summary())

print("\n=== Adaptation (the distilled lesson) ===")
if runtime.adaptations.impressions:
    for impression in runtime.adaptations.impressions:
        print(f"- {impression.name}: {impression.insight}")
else:
    print("No adaptation distilled yet.")

## 10. The full `Evidence` audit trail behind one decision

`Evidence` records which reasoning path was used, which policies were
checked, the composed plan, what `Recaller` recalled from memory, and
`Explainer`'s explanation — separately from the decision itself.

In [ ]:
import json

if runtime.memory.working:
    last_entry = runtime.memory.working[-1]
    print(json.dumps(
        {**last_entry.evidence.sources, "decision": last_entry.decision, "basis": last_entry.evidence.basis},
        indent=2, default=str,
    ))
else:
    print("No successful cycle was recorded -- every generated applicant breached a Policy.")

## Why this notebook reasons at runtime instead of hardcoding

- **Skills are prompts, not code.** The score/DTI banding and risk
  grading rules are written in plain English inside each `Skill`;
  `Runtime.reason()` threads the stacked skill prompts into the
  `Reasoner`, so no scoring function or decision grid is hand-written.
- **`Policy` is plain English, not code.** Each `Policy` is a single
  `statement` the LLM judges against the applicant's context — there is
  no hardcoded boolean expression anywhere. Without a model the
  statement-only policy is simply not applicable, rather than being
  faked with hand-written logic.
- **No hardcoded applicant data.** `GenerateApplicantScenario` invents
  every applicant from a one-line steer at call time; the fallback path
  varies applicants procedurally instead of reading a fixed literal dict.
- **DSPy and GEPA are used sparingly.** Only the genuinely judgment-laden
  stages (`Policy`, `Discoverer`, `Reasoner`, `Explainer`, and this
  notebook's scenario generator) carry a DSPy signature; `Selector`,
  `Composer` and `Scheduler` stay plain Python because there is no
  judgment call to make in deduplicating, flattening or copying a list.
  GEPA itself is reserved for `Reasoner.optimize_with_gepa()` alone — not
  invoked in this notebook, since optimizing a reasoning program is a
  development-time exercise, not a per-cycle one.
- **Every class is native English**: `ear.Runtime`, `ear.Policy`,
  `ear.ModelBinding`, and so on are real classes defined in `ear/`.